![logo](https://github.com/HelmholtzAI-Consultants-Munich/XAI-Tutorials/blob/main/docs/source/_figures/Helmholtz-AI.png?raw=true)

# Hands-On Session: Explainable AI for Random Forest Models

Building on the previously trained Random Forest classifier for **AML vs. non-AML** prediction, this notebook focuses on interpreting and analyzing the model’s decision-making process. Using RNA-seq gene expression data from [Warnat-Herresthal et al. (2020)](https://doi.org/10.1016/j.isci.2019.100780), we apply and compare multiple explainability methods to understand which genes drive the model’s predictions. The objectives of this notebook are to compute and compare model-agnostic and model-specific explanation methods, critically assess their differences, and reflect on the strengths and limitations of each approach in a high-dimensional, multi-study biomedical setting.

--------

## Getting Started

### Setup Colab environment

If you installed the packages and requirements on your own machine, you can skip this section and start from the import section.
Otherwise, you can follow and execute the tutorial on your browser. In order to start working on the notebook, click on the following button. This will open this page in the Colab environment and you will be able to execute the code on your own.

<a href="https://colab.research.google.com/github/HelmholtzAI-Consultants-Munich/XAI-Tutorials/blob/main/xai-for-random-forest/Bio-5-Tutorial_XAI_for_RandomForests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Now that you opened the notebook in Colab, follow the next step:

1. Run this cell to connect your Google Drive to Colab and install packages
2. Allow this notebook to access your Google Drive files. Click on 'Yes', and select your account.
3. "Google Drive for desktop wants to access your Google Account". Click on 'Allow'.
   
At this point, a folder has been created in your Drive and you can navigate it through the lefthand panel in Colab, you might also have received an email that informs you about the access on your Google Drive.

In [ ]:
# Mount drive folder to dbe abale to download repo
# from google.colab import drive
# drive.mount('/content/drive')

# Switch to correct folder'
# %cd /content/drive/MyDrive

In [ ]:
# Don't run this cell if you already cloned the repo 
# %rm -r XAI-Tutorials
# !git clone --branch main https://github.com/HelmholtzAI-Consultants-Munich/XAI-Tutorials.git

In [ ]:
# Install al required dependencies and package versions
# %cd XAI-Tutorials
# !pip install -r requirements_xai-for-random-forest.txt
# %cd xai-for-random-forest

### Imports

Let's start with importing all required Python packages.

In [ ]:
# Load the required packages

import os
import sys
import pickle

import numpy as np
import pandas as pd

import shap
from fgclustering import (
    forest_guided_clustering, 
    forest_guided_feature_importance, 
    plot_forest_guided_feature_importance, 
    plot_forest_guided_decision_paths,
    DistanceRandomForestProximity,
    ClusteringKMedoids,
)
from sklearn.metrics import get_scorer
from sklearn.inspection import permutation_importance

sys.path.append('./')
import utils

Now, we fix the random seeds to ensure reproducible results, as we work with (pseudo) random numbers.

In [ ]:
# assert reproducible random number generation
seed = 1
np.random.seed(seed)

## Dataset Description

The data originates from [Warnat-Herresthal et al. (2020)](https://doi.org/10.1016/j.isci.2019.100780), *"Scalable prediction of acute myeloid leukemia using high-dimensional machine learning and blood transcriptomics."*

The original study constructed a large multi-study dataset through a systematic search of the GEO database, followed by multiple filtering steps (cell type, species, platform, minimum sample size, and expert curation), as illustrated in teh figure below. After removal of duplicates and exclusion of studies using sorted cell populations, three platform-specific datasets were established.

In this hands-on session, we focus exclusively on **Dataset 3 (RNA-seq)**, which contains:

- **1,181 samples**
- **AML patients**
- **Non-AML samples**, including:
  - Healthy individuals  
  - Other leukemia subtypes (e.g., ALL, CLL, CML, MDS)  
  - Other non-leukemic diseases  

The dataset includes **log2-transformed, normalized gene expression values** for over **12,000 genes**. In the original study, all datasets were harmonized to a shared gene set (12,708 genes) to enable cross-platform comparisons. Each dataset was normalized independently per platform prior to downstream analysis. Gene expression profiling was performed on **Peripheral Blood Mononuclear Cells (PBMC)** and **Bone Marrow (BM)** samples. In the original study, these tissues were considered equivalent for AML diagnosis and analyzed jointly.

For this hands-on session, we use the normalized RNA-seq data as provided and focus on modelling and interpretability rather than preprocessing.

## Load Processed Data and Trained Model

In this step, we load the processed RNA-seq dataset and the previously trained Random Forest model. The trained model will serve as the basis for applying and comparing different explainability (XAI) methods.

In [ ]:
output_dir = "aml_case_study"

In [ ]:
# load previously stored data
with open(os.path.join(output_dir, "data_train_test.pickle"), "rb") as handle:
    data_train, data_test, gene_features = pickle.load(handle)

X_train = data_train[gene_features]
y_train = data_train["Condition"]
X_test = data_test[gene_features]
y_test = data_test["Condition"]

print(f"Number of training samples {X_train.shape[0]} ({round(X_train.shape[0]/(X_train.shape[0]+X_test.shape[0])*100)}%) with {sum(y_train=='0_Control')} control and {sum(y_train=='1_Cancer')} cancer samples.")
print(f"Number of testing samples {X_test.shape[0]} ({round(X_test.shape[0]/(X_train.shape[0]+X_test.shape[0])*100)}%) with {sum(y_test=='0_Control')} control and {sum(y_test=='1_Cancer')} cancer samples.")

In [ ]:
# load previously trained model
with open(os.path.join(output_dir, "rf_model.pickle"), "rb") as handle:
    rf_model = pickle.load(handle)

rf_model

## Now, what are our possibilities to interpret a Random Forest model?

First, we examine the distribution of gene–gene correlations to assess feature dependence in the dataset, as several explainability methods rely on (explicit or implicit) assumptions of feature independence.

In [ ]:
utils.plot_correlation_distribution(data_train)

In [ ]:
# we only want to plot the top features for sake of visualisation
top_n = 30

### Interpretation with Feature Importance Measures

#### Permutation Feature Importance

Recall, the Permutation Feature Importance is defined to be the decrease in a model score when a single feature value is randomly permuted. This procedure breaks the relationship between the feature and the target. Thus the drop in the model score is indicative of how much the model depends on the feature. Lets now apply it to our penguins dataset:

<font color='green'>

#### Task 1: Calculate permutation feature importance for the trained Random Forest model with the appropriate scorer.

*Note: Computing permutation feature importance can be computationally expensive, particularly in high-dimensional datasets such as RNA-seq data. We therefore recommend running this analysis in a separate notebook to avoid slowing down the remainig tasks. To speed up computations, you can set the n_repeats to 5 or lower.*

To compute permutation feature importances, we first need to define a scoring metric. In this case, we use "accuracy" to evaluate performance. The permutation procedure is then applied feature by feature, which requires repeatedly re-evaluating the model. Because each feature is permuted separately, the process can be computationally intensive, especially in high-dimensional settings.

In [ ]:
scorer = get_scorer("accuracy")
result = permutation_importance(rf_model, X_train, y_train, n_repeats=5, scoring=scorer, random_state=seed, n_jobs=8)
utils.plot_permutation_feature_importance(result=result, data=X_train, title="Permutation Feature Importance", figsize=(10,9), top_n=top_n)

#### Random Forest Feature Importance

An alternative to Permutation Feature Importance is the Random Forest specific feature importance method based on the mean decrease in impurity. The mean decrease in impurity is defined as the total decrease in node impurity averaged over all trees of the ensemble. This Feature Importances is directly provided by the fitted attribute `feature_importances_` .

Lets plot the feature importance based on mean decrease in impurity:

In [ ]:
utils.plot_impurity_feature_importance(rf_model.feature_importances_, names=X_train.columns, title="Random Forest Feature Importance", top_n=top_n, figsize=(10,9))

### Interpretation with SHAP

Recall, with SHAP we get contrastive explanations that compare the prediction with the average prediction. The global interpretations are consistent with the local explanations, since the Shapley values are the “atomic unit” of the global interpretations.

<font color='green'>

#### Task 2: Run SHAP on the trained Random Forest model for the training data with the appropriate explainer.

To apply SHAP to our model, we must select an appropriate explainer. Since we are interpreting a tree-based model (Random Forest), we use the `TreeExplainer()`, which is specifically optimized for tree ensembles. Because this is a classification task, we set the model output to "probability", so that SHAP values explain contributions in terms of predicted class probabilities rather than raw log-odds.

In [ ]:
# run SHAP
explainer = shap.TreeExplainer(model=rf_model, data=shap.maskers.Independent(X_train, max_samples=len(X_train)), model_output="probability")
shap_values = explainer(X_train)

# get original unscaled feature values
shap_values.data = X_train.round(4) 

#NOTE: the new SHAP package is still a bit buggy.
#      the returned dimensions for the shaply value
#      matrix are swapped. Hence, we need to put
#      then into correct order first.
shap_values.values = list(np.transpose(shap_values.values,(2,0,1)))

Remember that the SHAP values explain why a prediction for a single observation is different from the average prediction for all the observations in the data set. For our AML classifier, the shap explainer produces two expected values, corresponding to the average predicted probability for each class.

In [ ]:
# average prediction for the dataset
ev = explainer.expected_value
pm = np.mean(rf_model.predict_proba(X_train), axis=0)

print(f'Models average prediction for our data set is for class 0: {round(pm[0],3)}, for class 1: {round(pm[1],3)}')
print(f'Expected value for our data set is for class 0: {round(ev[0],3)}, for class 1: {round(ev[1],3)}')

<font color='green'>

#### Task 3: Plot the summary plot as barplot for the absolute SHAP values and the summary plot as a beeswarm plot. For multi-class calssification, the beeswarm plot has to be plotted per class.

To get a general overview on the features with the highest contributions, we can plot the average absolute SHAP values for each class.

In [ ]:
shap.summary_plot(
    shap_values.values, 
    shap_values.data, 
    plot_type='bar')

For a more granular view we can use the beeswarm or violin plots. Remember, those have to be plotted class-wise.

In [ ]:
# summary plot for class 0 = non-AML
target_class = 0
shap.summary_plot(
    shap_values.values[target_class], 
    shap_values.data)

In [ ]:
# summary plot for class 1 = AML
target_class = 1
shap.summary_plot(
    shap_values.values[target_class], 
    shap_values.data)

<font color='green'>

#### Task 4: Plot the force plots of the first patient in the dataset.

In [ ]:
observation_of_interest = 0
print(f'For observation of interest {observation_of_interest} the target value is {y_train[observation_of_interest]}; and models prediction for the observation of interest: {rf_model.predict(X_train)[observation_of_interest]}')

In [ ]:
target_class = 0
shap.force_plot(
    explainer.expected_value[target_class], 
    shap_values.values[target_class][observation_of_interest], 
    shap_values.data.iloc[observation_of_interest], 
    matplotlib=True)

In [ ]:
target_class = 1
shap.force_plot(
    explainer.expected_value[target_class], 
    shap_values.values[target_class][observation_of_interest], 
    shap_values.data.iloc[observation_of_interest], 
    matplotlib=True)

### Interpretation with Forest-Guided Clustering

Recall, FGC does not assume independence of model features, because it computes the feature importance based on subgroups of instances that follow similar decision rules within the Random Forest model.

<font color='green'>

#### Task 5: Run FGC on the trained Random Forest model.

We begin by determining the optimal number of clusters (k). FGC internally evaluates multiple values of k in the range 2–10 using stability-based clustering metrics (e.g., Jaccard Index) and selects the most stable solution. 

In [ ]:
# define number of clusters and obtain clustered data
fgc = forest_guided_clustering(
    k=(2,10),
    estimator=rf_model, 
    X=X_train, 
    y=y_train, 
    clustering_distance_metric=DistanceRandomForestProximity(), 
    clustering_strategy=ClusteringKMedoids(method="fasterpam", max_iter=100),
    JI_discart_value=0.9,
)

Here, the best solution is found at k = 8 clusters, indicating the model recognizes 8 meaningful subgroups in the decision space. 

#### Basic Analysis with FGC

After identifying the clusters derived from the Random Forest decision paths, we compute feature importance both at the cluster level (local importance) and across the full dataset (global importance). Importance is quantified using the Wasserstein distance, which measures differences between feature distributions across clusters.

These cluster-specific and global importance scores highlight which genes are most influential in separating distinct decision regions of the model, revealing how feature relevance varies across subpopulations rather than assuming a single global importance structure.

In [ ]:
# calculate feature importance on the full gene set
feature_importance = forest_guided_feature_importance(
    X=X_train, 
    y=y_train, 
    cluster_labels=fgc.cluster_labels,
    model_type=fgc.model_type,
    feature_importance_distance_metric="wasserstein",
)

The feature importance plot indicates that most clusters are characterized by distinct gene expression signatures among their top-ranking features. This suggests that different subgroups of samples are driven by partially unique gene patterns. Notably, three genes (BTN3A3, CD48, and GVINP1) appear among the top features in both AML-enriched and control-associated clusters, indicating that certain genes may contribute to multiple decision regions and potentially play context-dependent roles in the model’s classification.

In [ ]:
# plot top 30 globally important features
plot_forest_guided_feature_importance(
    feature_importance_local=feature_importance.feature_importance_local,
    feature_importance_global=feature_importance.feature_importance_global,
    top_n=top_n,
    num_cols=5,
)

Next, we visualize the decision path heatmap, which illustrates how gene expression patterns differ across clusters for the top-ranked features. Each cluster corresponds to a group of samples that follow similar decision paths within the Random Forest. The heatmap therefore provides insight into how distinct expression profiles map onto specific decision regions.

In [ ]:
# plot decision path plot for the top cancer genes and metadata
plot_forest_guided_decision_paths(
    data_clustering=feature_importance.data_clustering,
    model_type=fgc.model_type,
    top_n=top_n,
    distributions=False,
    heatmap=True,
)

The box plots allow us to examine the magnitude of expression differences between clusters and to assess the actual value ranges of the most influential features. By visualizing cluster-specific boxplots for the top genes, we gain deeper insight into how these features are distributed across the eight decision-path clusters and how strongly they differentiate specific subgroups.

In [ ]:
# plot decision path plot for the top cancer genes and metadata
plot_forest_guided_decision_paths(
    data_clustering=feature_importance.data_clustering,
    model_type=fgc.model_type,
    top_n=top_n,
    num_cols=9,
    distributions=True,
    heatmap=False,
)

### Advanced Analysis with FGC

To better understand what distinguishes AML (Acute Myeloid Leukemia) samples from non-AML samples, and to assess potential confounding effects, we focus on the 30 most informative genes specific to the AML-enriched cluster identified by Forest-Guided Clustering.

We first extract the top 30 features based on cluster-specific (local) feature importance from the AML cluster. We then extend this feature set by incorporating relevant metadata variables, including tissue type (e.g., PBMC, bone marrow), study origin (e.g., GSE identifiers), and disease labels (e.g., ALL, CLL, healthy). This combined feature set allows us to evaluate both biological signals and potential technical or cohort-related influences. Feature importance is recomputed using this enriched feature space to assess the joint contribution of gene expression and metadata variables. 

Finally, we visualize the resulting decision paths to examine how clusters are structured and to identify whether technical or biological covariates may act as confounding factors. This analysis helps ensure that the observed separation of AML samples reflects meaningful biological patterns rather than study-specific or technical artifacts, strengthening confidence in the robustness of the model’s decision logic.

In [ ]:
# select the top 30 locally important features for teh AML cluster
feature_importance_local = feature_importance.feature_importance_local
feature_importance_local.columns = [f"Cluster_{i+1}" for i in range(feature_importance_local.shape[1])]
feature_importance_local.sort_values(by="Cluster_1", ascending=False, inplace=True)

top_cancer_genes = list(feature_importance_local.index[:30])
selected_metadata = ["Tissue", "GSE", "Disease"]

In [ ]:
# calculate feature importance for the top cancer genes
feature_importance_top_cancer_genes = forest_guided_feature_importance(
    X=data_train[top_cancer_genes],
    y=data_train["Condition"], 
    cluster_labels=fgc.cluster_labels,
    model_type=fgc.model_type,
    feature_importance_distance_metric="wasserstein",
)

In [ ]:
# calculate feature importance for the metadata
feature_importance_metadata = forest_guided_feature_importance(
    X=data_train[selected_metadata],
    y=data_train["Condition"], 
    cluster_labels=fgc.cluster_labels,
    model_type=fgc.model_type,
    feature_importance_distance_metric="jensenshannon",
)

In [ ]:
data_clustering = pd.concat([feature_importance_metadata.data_clustering, feature_importance_top_cancer_genes.data_clustering], axis=1)
data_clustering = data_clustering.loc[:, ~data_clustering.columns.duplicated()]

The Forest-Guided Clustering analysis identifies eight distinct clusters, each representing groups of samples that follow similar decision paths within the trained Random Forest. Notably, Cluster 1 consists exclusively of AML samples, indicating that AML exhibits a highly distinct gene expression signature that the model captures consistently and robustly.

In contrast, non-AML samples are distributed across the remaining seven clusters (Clusters 2–8). These clusters show heterogeneous expression patterns, reflecting biological diversity among non-AML conditions and potentially the influence of technical or study-specific factors. This structured partitioning suggests that the model does not simply learn a single global AML versus non-AML boundary. Instead, it captures a more nuanced internal representation of the data.

Although the model was trained for binary classification, it implicitly organizes samples into meaningful subgroups based on shared decision logic. This layered structure highlights the capacity of tree-based models to encode complex biological patterns and may provide valuable insight for subtype exploration and for understanding model behavior in heterogeneous biomedical datasets.

In [ ]:
# plot decision path plot for the top cancer genes and metadata
plot_forest_guided_decision_paths(
    data_clustering=data_clustering,
    model_type=fgc.model_type,
    distributions=False,
    heatmap=True,
)

In [ ]:
# visualize metadata as stacked bar plots
utils.plot_stacked_bar_chart(feature_importance_metadata, ["GSE", "Disease", "Tissue"])

In [ ]:
feature_importance_top_X = forest_guided_feature_importance(
    X=data_train[list(feature_importance_local.index[:100])],
    y=data_train["Condition"], 
    cluster_labels=fgc.cluster_labels,
    model_type=fgc.model_type,
    feature_importance_distance_metric="wasserstein",
)

utils.plot_feature_importance_by_AML_cluster(feature_importance_top_X, figsize=(22,2))

#### Comparison to unsupervised clustering

To assess the added value of the supervised Forest-Guided Clustering (FGC), we compare its results to an unsupervised k-medoids clustering approach. For consistency, we use the same number of clusters (k) determined by FGC and apply k-medoids directly on the standardized gene expression data using Euclidean distance. This allows us to evaluate whether incorporating model-derived information improves subgroup discovery and alignment with known biological labels.

In [ ]:
# comparse results with k-medoids clustering heatmap with k=8
dataset = utils.kmedoids_clustering_heatmap(data_train, k=8, name="case_study_1")

The k-medoids clustering applied to the expression data without supervision reveals key differences compared to the Forest-Guided Clustering (FGC) results.

From the heatmap above we observe that:
- AML samples, which were cleanly grouped into a single distinct cluster using FGC, are now scattered across three different clusters (clusters 2, 3, and 4). Each of these clusters also contains non-AML samples, including MDS, CML, and ALL cases, reducing the specificity of AML separation.
- Other leukemia types (e.g., CLL, ALL, CML, MDS) are also not clearly separated into dedicated clusters, suggesting that unsupervised clustering fails to isolate leukemia types effectively.
- Several clusters (e.g., clusters 0, 1, 6, and 7) are instead dominated by non-leukemic diseases such as JIA, TB-vaccination, or Lyme disease. These disease-specific groupings indicate that stronger expression patterns unrelated to leukemia are driving the clustering structure.

This result highlights an important limitation of purely unsupervised approaches: The most prominent sources of variation in high-dimensional expression data may not align with the clinical or biological question of interest. In contrast, the supervised FGC approach leverages model-driven proximity, enabling more targeted discovery of biologically or clinically relevant subgroups—such as the AML-specific cluster. This underscores the value of integrating predictive models into clustering for goal-oriented subgroup discovery.

<font color='green'>

#### Question 1: What do you observe when comparing the different XAI methods?

## Conclusion: Comparing XAI Methods

In this notebook, we applied and compared multiple explainability methods to interpret the trained Random Forest model for AML vs. non-AML prediction. Although all methods aim to quantify feature relevance, they differ substantially in their assumptions, interpretability, sensitivity to feature dependence, and the type of information they provide.


### Permutation Feature Importance 

The permutation feature importance (PFI) results show near-zero importance values for most genes. Permuting individual features does not substantially decrease model performance. In high-dimensional RNA-seq data, many genes are strongly correlated and share overlapping biological signal. Because PFI permutes one feature at a time, correlated genes can compensate for the permuted feature, resulting in weak or diluted importance estimates. Thus, PFI tends to underestimate feature relevance in highly correlated settings.

PFI is model-agnostic and intuitive, but in redundant feature spaces it provides limited insight into how predictive signal is distributed across correlated gene groups.

### Random Forest Feature Importance

The built-in Random Forest feature importance (RFFI) produces a clear ranking of influential genes. However, RFFI does not explicitly account for feature correlation. When predictors are correlated, importance may be distributed across them depending on how often they are selected in tree splits. Additionally, RFFI can be biased toward features with higher variance or more splitting opportunities.


### SHAP

SHAP provides both global and local explanations of the Random Forest model. The global SHAP ranking shows substantial agreement with the RFFI, increasing confidence in the stability of the top-ranked genes. At the same time, SHAP decomposes each individual prediction into additive feature contributions, allowing us to quantify how much each gene pushes a prediction toward AML or non-AML.

The SHAP summary plots reveal the direction of effects (whether high or low expression increases AML probability), the variability of contributions across samples, and class-specific impact patterns. In contrast to PFI, SHAP distributes interaction effects across correlated features, making it more robust in redundant, high-dimensional settings. However, it still relies on assumptions about feature independence when estimating conditional expectations, so interpretations must remain cautious in strongly correlated feature spaces. Overall, SHAP provides richer insight than purely global ranking methods by linking global importance with local, directional effects.

### Forest-Guided Clustering

Forest-Guided Clustering (FGC) moves beyond individual feature attribution and instead examines the structure of the model’s decision process. By grouping samples that follow similar decision paths, FGC reveals that the Random Forest does not rely on a single global AML vs. non-AML rule, but instead partitions the data into distinct decision regions with partially different gene signatures.

Overlaying metadata onto clusters adds further interpretability. For example, clusters dominated by a single study may indicate batch effects, whereas the absence of tissue-driven clustering suggests that the model is not primarily separating bone marrow from PBMC samples. The presence of distinct clusters for different leukemia subtypes shows that the model captures biological structure beyond the binary label, implicitly distinguishing subtypes without being explicitly trained to do so.

FGC therefore provides subgroup-level interpretability and helps identify hidden structure and potential confounding factors. Because it leverages the internal tree structure rather than perturbing individual features, it is less sensitive to feature correlation and particularly informative in high-dimensional transcriptomic data.

### Final Remarks

Feature correlation plays a central role in interpreting explainability results in high-dimensional transcriptomic data. Redundant gene expression patterns weaken perturbation-based methods and distribute importance across correlated predictors.

No single XAI method provides a complete understanding of model behavior:

- **PFI** highlights limitations in correlated feature spaces.
- **RFFI** offers fast global ranking but may be biased.
- **SHAP** provides stable global and local explanations.
- **FGC** reveals structured heterogeneity, subgroup logic, and potential confounding effects.

Together, these complementary perspectives show that the Random Forest model captures not only AML vs. non-AML separation but also richer biological structure, including implicit discrimination among leukemia subtypes and study-level variation. Combining multiple XAI approaches provides a more comprehensive and reliable understanding of model behavior in complex, high-dimensional biomedical settings.
